In [2]:
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph,START,END
from langgraph.graph.message import add_messages

In [6]:
class State(TypedDict):
    messages: Annotated[list,add_messages]

graph_builder = StateGraph(State)
graph_builder

In [3]:
import os
from dotenv import load_dotenv
load_dotenv()

True

### Simple Chat-Bot

In [4]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.1-8b-instant"
)
llm

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.5.1', 'langchain': '1.3.14'}}, output_version=None, profile={'name': 'Llama 3.1 8B Instant', 'release_date': '2024-07-23', 'last_updated': '2024-07-23', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 131072, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000001592758B230>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000015927728050>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [7]:
def chatbot(state: State):
    return {"messages": [llm.invoke(state["messages"])]}

In [8]:
## Adding Node
graph_builder.add_node("llmchatbot",chatbot)

## Adding Edges
graph_builder.add_edge(START,"llmchatbot")
graph_builder.add_edge("llmchatbot",END)

## Compiling the graph
graph = graph_builder.compile()

In [9]:
response = graph.invoke({"messages":"Hi"})
response

{'messages': [HumanMessage(content='Hi', additional_kwargs={}, response_metadata={}, id='4521378e-6d19-4d7a-94c3-18e48b7384dd'),
  AIMessage(content='How can I help you today?', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 8, 'prompt_tokens': 36, 'total_tokens': 44, 'completion_time': 0.020167237, 'completion_tokens_details': None, 'prompt_time': 0.001791208, 'prompt_tokens_details': None, 'queue_time': 0.053820071, 'total_time': 0.021958445}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019fa85d-5de2-7c43-a59b-64163b4e4fd7-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 36, 'output_tokens': 8, 'total_tokens': 44})]}

In [10]:
response["messages"][-1].content

'How can I help you today?'

In [11]:
for event in graph.stream({"messages":"Hi, how are you?"}):
    for value in event.values():
        print(value["messages"][-1].content)

I'm functioning properly. How can I assist you today?


### Chat-Bot With Tools

In [12]:
from langchain_tavily import TavilySearch
tavily = TavilySearch(max_results=2)
tavily.invoke("What is the capital of France?")

{'query': 'What is the capital of France?',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://studyx.ai/questions/4ln09wo/3-what-is-the-capital-of-france',
   'title': 'Capital of France',
   'content': "+ Reason 1: Paris is the capital city of France. It is also its largest city.\n  + Reason 2: Paris is the center of French politics, culture, and commerce. The French President resides in Paris.\n\n  Summary\n\n  + The capital of France is Paris because it serves as the country's political, cultural, and economic hub.\n\n  \n\nAsk your next question\n\n Upload\n Text\n\nPaste or type your question to get answers\n\nicon-upload\n\nUpload Image or PDF to solve questions in it\n\nCtrl + V to paste\n\nSelect file [...] View solution\n\nQuestion 5\n\nQuestion What is the capital of France - A) Berlin - B) Madrid - C) Paris - D) Rome\n\nExpert-verified Solution\n\nView solution\n\nQuestion 6\n\nTime Left 5 seconds What is the capital of France Paris 

In [13]:
from langchain_community.tools import ArxivQueryRun,WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper,ArxivAPIWrapper

C:\Users\CSMBD\AppData\Local\Temp\ipykernel_24284\1871147362.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import ArxivQueryRun,WikipediaQueryRun


In [14]:
api_wrapper_arxiv = ArxivAPIWrapper(top_k_results=2,doc_content_chars_max=500)
arxiv = ArxivQueryRun(api_wrapper=api_wrapper_arxiv,description="Query arxiv papers")

In [15]:
arxiv.invoke("What are the latest papers on quantum computing?")

'Published: 2025-12-16\nTitle: Tianyan: Cloud services with quantum advantage\nAuthors: Tianyan Quantum Group\nSummary: Tianyan Quantum Cloud Platform offers cloud services demonstrating quantum advantage capabilities with a Zuchongzhi 3.0-like superconducting quantum processor. This cloud-accessible superconducting quantum prototype, named Tianyan-287, features 105 qubits and achieves high operational fidelities, with single-qubit gates, two-qubit gates, and readout fidelity at 99.90%, 99.56%, 98.7'

In [16]:
api_wrapper_wiki = WikipediaAPIWrapper(top_k_results=2,doc_content_chars_max=500)
wiki = WikipediaQueryRun(api_wrapper=api_wrapper_wiki)

In [17]:
tools = [arxiv,wiki,tavily]

In [18]:
## Initializing the LLM Model
from langchain_groq import ChatGroq
llm = ChatGroq(model="qwen/qwen3.6-27b")

In [19]:
llm.invoke("What is AI?")

AIMessage(content='\n<think>\nHere\'s a thinking process:\n\n1.  **Understand User Query**: The user asks "What is AI?" This is a fundamental, open-ended question about Artificial Intelligence.\n\n2.  **Identify Key Components of AI Definition**:\n   - Core definition: Simulation of human intelligence by machines\n   - Key capabilities: Learning, reasoning, problem-solving, perception, language understanding\n   - Types/Categories: Narrow AI vs. General AI (and sometimes Super AI, though theoretical)\n   - How it works: Algorithms, data, machine learning, neural networks\n   - Applications: Everyday examples (recommendations, assistants, autonomous vehicles, etc.)\n   - Current state: Mostly narrow/specialized AI, rapidly advancing\n   - Important context: Not conscious, no true understanding, relies on patterns in data\n\n3.  **Structure the Response**:\n   - Clear, concise definition first\n   - Break down key aspects (how it works, types, examples)\n   - Clarify common misconception

In [32]:
llm_with_tools = llm.bind_tools(tools=tools)
llm_with_tools.invoke("Find paper 'Attention all you need'")

AIMessage(content='', additional_kwargs={'reasoning_content': 'The user is asking for a specific paper titled "Attention all you need".\nI should search for this paper on Arxiv to get the details and abstract.\nThe query should be exactly the title of the paper.\n\n1.  **Identify the tool**: `arxiv` is the most appropriate tool for finding academic papers.\n2.  **Formulate the query**: "Attention Is All You Need" (Note: The user wrote "Attention all you need", but the famous paper is usually titled "Attention Is All You Need". I will search for the user\'s query first, but correct the capitalization or likely missing word if needed. Actually, searching for "Attention all you need" should find it, or "Attention Is All You Need" is safer if I know that\'s the famous one. I\'ll stick to the user\'s input or the standard title. The standard title is "Attention Is All You Need". I will search for the exact phrase the user provided, and maybe the correct title if that fails, but Arxiv search

In [33]:
from langgraph.graph import StateGraph,START,END
from langgraph.prebuilt import ToolNode
from langgraph.prebuilt import tools_condition

def tool_calling_llm(state: State):
    return {"messages": [llm_with_tools.invoke(state["messages"])]}

builder = StateGraph(State)
builder.add_node("tool_calling_llm", tool_calling_llm)
builder.add_node("tools", ToolNode(tools=tools))

builder.add_edge(START, "tool_calling_llm")
builder.add_conditional_edges("tool_calling_llm", tools_condition)
builder.add_edge("tools", END)

graph = builder.compile()

In [43]:
response = graph.invoke({"messages":"Who is Cr7 ans find paper 'Attention all you need'?"})

In [46]:
response

{'messages': [HumanMessage(content="Who is Cr7 ans find paper 'Attention all you need'?", additional_kwargs={}, response_metadata={}, id='4ebd4310-06cc-439d-9f7c-d10977597da1'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'Thinking Process:\n\n1.  **Identify the entities and requests:**\n    *   Entity 1: "Cr7" (Cristiano Ronaldo).\n    *   Entity 2: Paper titled \'Attention all you need\' (Attention Is All You Need).\n    *   Task: Identify who Cr7 is and find the specified paper.\n\n2.  **Analyze "Cr7":**\n    *   Cr7 refers to Cristiano Ronaldo, a famous Portuguese professional footballer.\n    *   I can use the `wikipedia` tool to get a concise summary or just rely on internal knowledge, but using the tool is safer for accuracy and providing a good summary.\n\n3.  **Analyze "Attention all you need":**\n    *   The user is asking for the paper "Attention Is All You Need".\n    *   This is a famous paper in the field of Artificial Intelligence/Machine Learning, in

In [50]:
response["messages"][-1].content

"Page: Cristiano Ronaldo\nSummary: Cristiano Ronaldo dos Santos Aveiro (born 5 February 1985) is a Portuguese professional footballer who plays as a forward for and captains the Saudi Pro League club Al-Nassr and the Portugal national team. Nicknamed CR7, he is widely regarded as one of the greatest players in history, having won numerous individual accolades throughout his career including five Ballon d'Or awards, a record three UEFA Men's Player of the Year Awards, and four European Golden Shoes"

In [45]:
for val in response["messages"]:
    val.pretty_print()

================================ Human Message =================================

Who is Cr7 ans find paper 'Attention all you need'?
================================== Ai Message ==================================
Tool Calls:
  wikipedia (fmzwrhb65)
 Call ID: fmzwrhb65
  Args:
    query: Cristiano Ronaldo
================================= Tool Message =================================
Name: wikipedia

Page: Cristiano Ronaldo
Summary: Cristiano Ronaldo dos Santos Aveiro (born 5 February 1985) is a Portuguese professional footballer who plays as a forward for and captains the Saudi Pro League club Al-Nassr and the Portugal national team. Nicknamed CR7, he is widely regarded as one of the greatest players in history, having won numerous individual accolades throughout his career including five Ballon d'Or awards, a record three UEFA Men's Player of the Year Awards, and four European Golden Shoes


In [ ]:
# Agentic AI With Langgraph And MCP Crash Course-Part 1 -- 1:20min